# PIPELINE FRAMEWORK FOR EMPLOYER RECORD MATCHING & MERGING 
### USING CHARACTER_BASED, TOKEN BASED AND SEMANTIC_BASED SIMILARITY MODELS

In [17]:
import pandas as pd

# Load your Excel or CSV file
df = pd.read_excel("C:\\Users\\USER\\Desktop\\Data cleaning (NLP)\\NLP BERT TEST 2.xlsx")
df

,E NO,E NAME,E NO 2,E NAME 2,KOD,STATE,BP 2024_06,BP 2024_05,BP 2024_04,BP 2024_03,BP 2024_02,BP 2024_01,Unnamed: 12,Unnamed: 13
0,NaN,"""K"" LINE LOGISTICS (M) SDN. BHD.",NaN,K LINE LOGISTICS MALAYSIA SDN BHD,E-05,Selangor,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,940960-A,&SAMHOUD SDN BHD,940960A,SAMHOUD SDN BHD,E-11,W.P. Kuala Lumpur,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,940960A,&SAMHOUD SDN BHD,940960A,SAMHOUD SDN BHD,E-11,W.P. Kuala Lumpur,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0021,(BHG.PERANCANGAN & PENYELIDIKAN),21,BHGPERANCANGAN PENYELIDIKAN,N.E.C.,W.P. Putrajaya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LWS/21/2002,(D.U.N) ENTERPRISE,LWS212002,DUN ENTERPRISE,N.E.C.,Sarawak,4.0,4.0,4.0,4.0,4.0,4.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,115470-H,'TA' ELECTRICAL SALE AND SERVICE,115470H,TA ELECTRICAL SALE AND SERVICE,E-16_17,Perak,4.0,4.0,4.0,4.0,4.0,4.0,NaN,NaN
3996,821005095239,'UMAIR BIN KASA @ HALIMI,821005095239,UMAIR BIN KASA HALIMI,E-16_17,Kedah,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN
3997,000016083-P,'Y' HAIR DRESING SALOON,16083P,Y HAIR DRESING SALOON,E-16_17,W.P. Kuala Lumpur,4.0,4.0,4.0,6.0,6.0,6.0,NaN,NaN
3998,000160863P,'Y' HAIR DRESING SALOON,160863P,Y HAIR DRESING SALOON,E-16_17,W.P. Kuala Lumpur,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Handle missing values & convert to string
df["E NAME"] = df["E NAME"].fillna("").astype(str)

In [3]:
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# PIPELINE FRAMEWORK FOR EMPLOYER RECORD MATCHING & MERGING 
### USING CHARACTER_BASED, TOKEN BASED AND SEMANTIC_BASED SIMILARITY MODELS

In [4]:
import pandas as pd
import re
from fuzzywuzzy import fuzz
from Levenshtein import distance as levenshtein_distance, jaro_winkler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
import os
from datetime import datetime

# -----------------------------
# Model-specific thresholds
# -----------------------------
THRESHOLDS = {
    "FUZZY": 98,
    "JARO_WINKLER": 98,
    "LEVENSHTEIN": 95,
    "JACCARD": 100,
    "TFIDF": 51,
    "BERT": 88,
    "ENSEMBLE": 96
}

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# -----------------------------
# Helper function: Normalize name
# -----------------------------
def normalize_name(name):
    """Normalize name for comparison"""
    if pd.isna(name):
        return ""
    return str(name).lower().strip()

# -----------------------------
# 1. Clean unwanted company suffixes
# -----------------------------
def clean_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    # Remove common business suffixes
    patterns = [
        r"\bsdn\.?\s*bhd\.?\b",
        r"\bsendiran\s*berhad\b",
        r"\bs/b\b",
        r"\bsdn\b",
        r"\bbhd\b",
    ]

    for p in patterns:
        name = re.sub(p, "", name)

    # Remove extra spaces
    name = re.sub(r"\s+", " ", name).strip()
    
    # REMOVE punctuation: . / ( ) , -
    name = re.sub(r"[./():,\-&@]", " ", name)

    return name

def clean_employer_id(eid):
    if pd.isna(eid):
        return ""

    eid = str(eid).lower().strip()

    # 🔹 Remove punctuation from ID
    eid = re.sub(r"[.,()/\-]", "", eid)

    # Remove extra spaces
    eid = re.sub(r"\s+", "", eid)

    return eid

# -----------------------------
# 2. Tokenize → Lemmatize → Stem
# -----------------------------
def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    tokens = word_tokenize(text)

    # Lemmatize FIRST → keeps meaning
    lemmas = [lemmatizer.lemmatize(word) for word in tokens]

    # Stem SECOND → reduce variations
    stems = [stemmer.stem(word) for word in lemmas]

    # Rejoin
    return " ".join(stems)

# -----------------------------
# Individual similarity functions - ALL RETURN 0-100 SCALE
# -----------------------------
def fuzzy_similarity(name1, name2):
    """Return fuzzy similarity score (0-100)"""
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)
    return fuzz.ratio(n1, n2)  # Already 0-100

def levenshtein_similarity(name1, name2):
    """Return Levenshtein similarity score (0-100)"""
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)
    
    distance = levenshtein_distance(n1, n2)
    max_len = max(len(n1), len(n2))
    
    if max_len == 0:
        return 0.0
    
    # Convert 0-1 to 0-100
    return (1 - (distance / max_len)) * 100

def jaccard_similarity(name1, name2):
    """Return Jaccard similarity score (0-100)"""
    n1 = set(normalize_name(name1).split())
    n2 = set(normalize_name(name2).split())
    
    if not n1 or not n2:
        return 0.0
    
    # Convert 0-1 to 0-100
    return (len(n1.intersection(n2)) / len(n1.union(n2))) * 100

def jaro_winkler_similarity(name1, name2):
    """Return Jaro-Winkler similarity score (0-100)"""
    n1, n2 = normalize_name(name1), normalize_name(name2)
    # Convert 0-1 to 0-100
    return jaro_winkler(n1, n2) * 100

def tfidf_cosine_similarity(name1, name2):
    """Return TF-IDF cosine similarity score (0-100)"""
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)
    
    tfidf = TfidfVectorizer().fit([n1, n2])
    vectors = tfidf.transform([n1, n2])
    
    score = cosine_similarity(vectors[0], vectors[1])[0][0]
    # Convert 0-1 to 0-100
    return score * 100

# Initialize BERT model
try:
    bert_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    BERT_AVAILABLE = True
except Exception as e:
    print(f"⚠️ BERT model not available: {e}")
    print("⚠️ Continuing without BERT similarity...")
    BERT_AVAILABLE = False

def bert_similarity(name1, name2):
    """Return BERT cosine similarity score (0-100)"""
    if not BERT_AVAILABLE:
        return 0.0
    
    n1 = normalize_name(name1)
    n2 = normalize_name(name2)
    
    embeddings = bert_model.encode([n1, n2])
    similarity = np.dot(embeddings[0], embeddings[1]) / (
        np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])
    )
    
    # Convert 0-1 to 0-100
    return float(similarity) * 100

# -----------------------------
# Ensemble scoring - ALL INPUTS ARE 0-100
# -----------------------------
def ensemble_score(fz, jc, lev, jw, tfidf, bert):
    """Calculate weighted ensemble score (0-100)"""
    # All inputs are already 0-100 scale
    return round(
        (fz * 0.70) +
        (jw * 0.20) +
        ((tfidf+bert)/2) * 0.10, 2
    )

# -----------------------------
# Main similarity function
# -----------------------------
def are_similar(name1, name2):
    fz = fuzzy_similarity(name1, name2)
    jc = jaccard_similarity(name1, name2)
    lev = levenshtein_similarity(name1, name2)
    jw = jaro_winkler_similarity(name1, name2)
    tfidf = tfidf_cosine_similarity(name1, name2)
    bert = bert_similarity(name1, name2)

    ens = ensemble_score(fz, jc, lev, jw, tfidf, bert)

    is_match = (
        (fz >= THRESHOLDS["FUZZY"]) or
        (jw >= THRESHOLDS["JARO_WINKLER"]) or
        (lev >= THRESHOLDS["LEVENSHTEIN"]) or
        (jc >= THRESHOLDS["JACCARD"]) or
        (tfidf >= THRESHOLDS["TFIDF"]) or
        (bert >= THRESHOLDS["BERT"]) or
        (ens >= THRESHOLDS["ENSEMBLE"])
    )

    return is_match, fz, jc, lev, jw, tfidf, bert, ens

#Model-specific similarity thresholds were applied to accommodate the different 
#scoring behaviors of character-based, token-based, and semantic similarity models. 
#High thresholds were used for edit-distance metrics (Fuzzy, Levenshtein, Jaro–Winkler), 
#while moderate thresholds were adopted for Jaccard and TF-IDF to account for vocabulary variation. 
#A higher threshold was applied to BERT to ensure strong semantic alignment. An ensemble threshold 
#was used to capture consensus across models.

# -----------------------------
# Matching & merging process
# -----------------------------
def safe_average(values):
    """
    Safely compute average for a list of similarity scores.
    Returns 0 if the list is empty.
    """
    if not values:
        return 0.0
    return sum(values) / len(values)

def match_and_merge(df):
    merged_data = []
    seen_global = set()

    print("Preprocessing names...")
    df["E NAME"] = df["E NAME"].apply(clean_name).apply(preprocess_text)
    df["E NO"] = df["E NO"].apply(clean_employer_id)
    
    total_rows = len(df)
    processed = 0
    
    for state, group in df.groupby("STATE"):
        group = group.reset_index(drop=True)

        for i in range(len(group)):
            if (state, i) in seen_global:
                continue

            current_row = group.iloc[i]


            # Anchor row
            matches = [{
                "row": current_row,
                "scores": {
                    "fz":0, "jc": 0, "lev": 0, "jw": 0,
                    "tfidf": 0, "bert": 0 if BERT_AVAILABLE else 0.0,
                    "ens": 0
                }
            }]
            local_indices = [i]  # Track used indices

            # -----------------------------
            # Compare with remaining rows in group
            # -----------------------------
            for j in range(i + 1, len(group)):
                if (state, j) in seen_global:
                    continue

                candidate_row = group.iloc[j]

                is_match, fz, jc, lev, jw, tfidf, bert, ens = are_similar(
                    current_row["E NAME"], candidate_row["E NAME"]
                )

                e_no_match = (
                    str(current_row.get("E NO", "")).strip() ==
                    str(candidate_row.get("E NO", "")).strip()
                )

                if is_match or e_no_match:
                    matches.append({
                        "row": candidate_row,
                        "scores": {
                            "fz": fz, "jc": jc, "lev": lev, "jw": jw,
                            "tfidf": tfidf, "bert": bert, "ens": ens
                        }
                    })
                    local_indices.append(j)

            # -----------------------------
            # Filter weak rows (keep only strong ones)
            # -----------------------------
            anchor = matches[0]["row"]
            strong_rows = [anchor]  # always keep anchor
            weak_rows = []

            for m in matches[1:]:
                scores = m["scores"]
                strong_models = sum([
                    scores["fz"] >= THRESHOLDS["FUZZY"],
                    scores["jc"] >= THRESHOLDS["JACCARD"],
                    scores["lev"] >= THRESHOLDS["LEVENSHTEIN"],
                    scores["jw"] >= THRESHOLDS["JARO_WINKLER"],
                    scores["tfidf"] >= THRESHOLDS["TFIDF"],
                    scores["bert"] >= THRESHOLDS["BERT"],
                    scores["ens"] >= THRESHOLDS["ENSEMBLE"]
                ])
                if strong_models >= 1:  # flexible rule
                    strong_rows.append(m["row"])
                else:
                    weak_rows.append(m["row"])# Output weak rows as standalone

                    
            # -----------------------------
            # Build merged row or single row
            # -----------------------------
            if len(strong_rows) > 1:
                similar_df = pd.DataFrame(strong_rows)

                # Compute min scores for merged group (excluding anchor)
                min_fuzzy = min([m["scores"]["fz"] for m in matches[1:]] or [0])
                min_jaccard = min([m["scores"]["jc"] for m in matches[1:]] or [0])
                min_lev = min([m["scores"]["lev"] for m in matches[1:]] or [0])
                min_jaro = min([m["scores"]["jw"] for m in matches[1:]] or [0])
                min_tfidf = min([m["scores"]["tfidf"] for m in matches[1:]] or [0])
                min_bert = min([m["scores"]["bert"] for m in matches[1:]] or [0])
                min_ensemble = min([m["scores"]["ens"] for m in matches[1:]] or [0])

                merged_row = {
                    "STATE": state,
                    "E NO": ", ".join(sorted(set(similar_df["E NO"].astype(str)))),
                    "E NAME": ", ".join(sorted(set(similar_df["E NAME"].astype(str)))),
                    "KOD": ", ".join(sorted(set(similar_df["KOD"].astype(str)))),
                    "BP 2024_01": similar_df["BP 2024_01"].sum(),
                    "BP 2024_02": similar_df["BP 2024_02"].sum(),
                    "BP 2024_03": similar_df["BP 2024_03"].sum(),
                    "BP 2024_04": similar_df["BP 2024_04"].sum(),
                    "BP 2024_05": similar_df["BP 2024_05"].sum(),
                    "BP 2024_06": similar_df["BP 2024_06"].sum(),
                    "MERGE_COUNT": len(similar_df),
                    "MIN_FUZZY": min_fuzzy,
                    "MIN_JACCARD": min_jaccard,
                    "MIN_LEVENSHTEIN": min_lev,
                    "MIN_JARO_WINKLER": min_jaro,
                    "MIN_TFIDF": min_tfidf,
                    "MIN_BERT": min_bert,
                    "MIN_ENSEMBLE": min_ensemble
                }
                merged_data.append(merged_row)
            else:
                # Single row (no merge)
                row_dict = anchor.to_dict()
                row_dict.update({
                    "MERGE_COUNT": 1,
                    "MIN_FUZZY": 0,
                    "MIN_JACCARD": 0,
                    "MIN_LEVENSHTEIN": 0,
                    "MIN_JARO_WINKLER": 0,
                    "MIN_TFIDF": 0,
                    "MIN_BERT": 0 if BERT_AVAILABLE else 0.0,
                    "MIN_ENSEMBLE": 0
                })
                merged_data.append(row_dict)     
           
           # -----------------------------
           # Output weak rows as standalone
           # -----------------------------
            for wr in weak_rows:
                row_dict = wr.to_dict()
                row_dict.update({
                    "MERGE_COUNT": 1,
                    "MIN_FUZZY": 0,
                    "MIN_JACCARD": 0,
                    "MIN_LEVENSHTEIN": 0,
                    "MIN_JARO_WINKLER": 0,
                    "MIN_TFIDF": 0,
                    "MIN_BERT": 0 if BERT_AVAILABLE else 0.0,
                    "MIN_ENSEMBLE": 0
                })
                merged_data.append(row_dict)


            # -----------------------------
            # Mark as seen
            # -----------------------------
            for idx in local_indices:
                seen_global.add((state, idx))

            processed += 1
            if processed % 100 == 0:
                print(f"Progress: {processed}/{total_rows}")

    print("✅ Processing complete (NO ROW LOSS GUARANTEED)")
    return pd.DataFrame(merged_data)


# -----------------------------
# Function to create summary report
# -----------------------------
def create_summary_report(original_df, merged_df):
    """Create a summary report of the merging process"""
    
    summary = {
        "Process Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Thresholds Used": str(THRESHOLDS),
        "Original Row Count": len(original_df),
        "Merged Row Count": len(merged_df),
        "Reduction Percentage": f"{(1 - len(merged_df)/len(original_df)) * 100:.1f}%",
        "Average Merge Count": f"{merged_df['MERGE_COUNT'].mean():.2f}",
        "Total Merges": int(merged_df['MERGE_COUNT'].sum() - len(merged_df)),  # Subtract single rows
        "Single Rows (No Merge)": int((merged_df['MERGE_COUNT'] == 1).sum()),
        "Merged Groups": int((merged_df['MERGE_COUNT'] > 1).sum()),
    }
    
    # Calculate average similarity scores for merged rows only
    merged_only = merged_df[merged_df['MERGE_COUNT'] > 1]
    if len(merged_only) > 0:
        score_columns = ['MIN_FUZZY', 'MIN_JACCARD', 'MIN_LEVENSHTEIN', 
                        'MIN_JARO_WINKLER', 'MIN_TFIDF', 'MIN_ENSEMBLE']
        
        score_summary = {}
        for col in score_columns:
            if col in merged_only.columns:
                score_summary[f"Min {col[4:]}"] = f"{merged_only[col].mean():.1f}"
        
        summary.update(score_summary)
    
    return summary


# -----------------------------
# Function to save Excel file with timestamp
# -----------------------------
def save_to_excel(df, base_filename="merged_output", folder_path=None):
    """
    Save DataFrame to Excel and CSV with timestamp in filename.
    Returns the Excel file path if successful, else None.
    """

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{base_filename}_{timestamp}.xlsx"

    if folder_path:
        os.makedirs(folder_path, exist_ok=True)
        filepath = os.path.join(folder_path, filename)
    else:
        filepath = filename

    try:
        df.to_excel(filepath, index=False)

        csv_filepath = filepath.replace(".xlsx", ".csv")
        df.to_csv(csv_filepath, index=False, encoding="utf-8-sig")

        print(f"✅ File saved: {filepath}")
        print(f"✅ CSV saved: {csv_filepath}")

        return filepath

    except Exception as e:
        print(f"❌ Error saving file: {e}")
        return None
    
    
def remove_duplicate_ids_in_cell(e_no):
    """
    Remove duplicate employer IDs inside a comma-separated cell
    WITHOUT removing the row.
    """
    if pd.isna(e_no):
        return e_no

    ids = [x.strip() for x in str(e_no).split(",") if x.strip()]
    unique_ids = sorted(set(ids))
    return ", ".join(unique_ids) 

def remove_duplicate_names_in_cell(e_name):
    if pd.isna(e_name):
        return e_name

    names = [x.strip() for x in str(e_name).split(",") if x.strip()]
    unique_names = sorted(set(names))
    return ", ".join(unique_names)

merged_df = match_and_merge(df)
merged_df["E NAME"] = merged_df["E NAME"].apply(remove_duplicate_names_in_cell)
    # 🔹 CLEAN DUPLICATE IDS INSIDE E NO (POST-MERGE)
merged_df["E NO"] = merged_df["E NO"].apply(remove_duplicate_ids_in_cell)



# -----------------------------
# Main execution
# -----------------------------

if __name__ == "__main__":
    # Configuration
    INPUT_FILE = r"C:\Users\USER\Desktop\Data cleaning (NLP)\NLP BERT TEST 2.xlsx"
    OUTPUT_FOLDER = r"C:\Users\USER\Desktop\Data cleaning (NLP)\Output"
    
    print("=" * 60)
    print("EMPLOYER NAME DEDUPLICATION AND MERGING TOOL")
    print("=" * 60)
    
    # Load data
    print(f"\n📂 Loading data from: {INPUT_FILE}")
    try:
        df = pd.read_excel(INPUT_FILE)
        print(f"   ✓ Loaded {len(df)} rows")
        print(f"   ✓ Columns: {', '.join(df.columns.tolist())}")
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        exit(1)
    
    # Check required columns
    required_columns = ["STATE", "E NO", "E NAME"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"❌ Missing required columns: {missing_columns}")
        exit(1)
    
    # Run matching and merging
    print(f"\n🔍 Starting matching and merging process...")
    print(f"   Model-specific thresholds: {THRESHOLDS}")
    print(f"   BERT available: {'Yes' if BERT_AVAILABLE else 'No'}") 

    
    # Create summary report
    print(f"\n📊 Generating summary report...")
    summary = create_summary_report(df, merged_df)
    
    print("\n" + "=" * 60)
    print("PROCESS SUMMARY")
    print("=" * 60)
    for key, value in summary.items():
        print(f"{key:30}: {value}")
    
    # Save the merged data
    print(f"\n💾 Saving results...")
    saved_file = save_to_excel(
        merged_df, 
        base_filename="employer_merged_resultss",
        folder_path=OUTPUT_FOLDER
    )
    
    # Save summary as separate file
    if saved_file:
        summary_df = pd.DataFrame([summary])
        summary_file = saved_file.replace(".xlsx", "_summary.xlsx")
        summary_df.to_excel(summary_file, index=False)
        print(f"✅ Summary saved: {summary_file}")
    
        # Also save a detailed comparison file (optional)
        print(f"\n📋 Creating detailed score breakdown...")
        
        # Create a simplified version with key metrics
        simplified_cols = [
            "STATE", "E NO", "E NAME", "MERGE_COUNT", 
            "MIN_FUZZY", "MIN_JARO_WINKLER", "MIN_ENSEMBLE"
        ]
        
        # Keep only columns that exist
        available_cols = [col for col in simplified_cols if col in merged_df.columns]
        simplified_df = merged_df[available_cols]
        
        # Save simplified version
        simple_file = saved_file.replace(".xlsx", "_simplified.xlsx")
        simplified_df.to_excel(simple_file, index=False)
        print(f"✅ Simplified version saved: {simple_file}")
        
        print("\n" + "=" * 60)
        print(f"✅ PROCESS COMPLETED SUCCESSFULLY!")
        print("=" * 60)
        print(f"\n📁 Files created:")
        print(f"   1. Main results: {saved_file}")
        print(f"   2. Summary report: {summary_file}")
        print(f"   3. Simplified view: {simple_file}")
        print(f"   4. CSV version: {saved_file.replace('.xlsx', '.csv')}")
        
        # Show first few rows of results
        print(f"\n👀 Preview of merged results (first 5 rows):")
        print("-" * 60)
        display_cols = ["STATE", "E NAME", "MERGE_COUNT", "MIN_FUZZY", "MIN_ENSEMBLE"]
        display_cols = [col for col in display_cols if col in merged_df.columns]
        print(merged_df[display_cols].head().to_string(index=False))
        
    else:
        print("❌ Failed to save results")

C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Preprocessing names...
Progress: 100/4000
Progress: 200/4000
Progress: 300/4000
Progress: 400/4000
Progress: 500/4000
Progress: 600/4000
Progress: 700/4000
Progress: 800/4000
Progress: 900/4000
Progress: 1000/4000
Progress: 1100/4000
Progress: 1200/4000
Progress: 1300/4000
Progress: 1400/4000
Progress: 1500/4000
Progress: 1600/4000
Progress: 1700/4000
Progress: 1800/4000
Progress: 1900/4000
Progress: 2000/4000
Progress: 2100/4000
Progress: 2200/4000
Progress: 2300/4000
Progress: 2400/4000
Progress: 2500/4000
Progress: 2600/4000
Progress: 2700/4000
Progress: 2800/4000
Progress: 2900/4000
Progress: 3000/4000
Progress: 3100/4000
Progress: 3200/4000
Progress: 3300/4000
Progress: 3400/4000
✅ Processing complete (NO ROW LOSS GUARANTEED)
EMPLOYER NAME DEDUPLICATION AND MERGING TOOL

📂 Loading data from: C:\Users\USER\Desktop\Data cleaning (NLP)\NLP BERT TEST 2.xlsx
   ✓ Loaded 4000 rows
   ✓ Columns: E NO, E NAME, E NO 2, E NAME 2, KOD, STATE, BP 2024_06, BP 2024_05, BP 2024_04, BP 2024_03, B

## Prove that all the 4000 rows included in the final output

In [6]:
merged_df = match_and_merge(df)

print("Input rows:", len(df))
print("Accounted rows:", merged_df["MERGE_COUNT"].sum())


Preprocessing names...
Progress: 100/4000
Progress: 200/4000
Progress: 300/4000
Progress: 400/4000
Progress: 500/4000
Progress: 600/4000
Progress: 700/4000
Progress: 800/4000
Progress: 900/4000
Progress: 1000/4000
Progress: 1100/4000
Progress: 1200/4000
Progress: 1300/4000
Progress: 1400/4000
Progress: 1500/4000
Progress: 1600/4000
Progress: 1700/4000
Progress: 1800/4000
Progress: 1900/4000
Progress: 2000/4000
Progress: 2100/4000
Progress: 2200/4000
Progress: 2300/4000
Progress: 2400/4000
Progress: 2500/4000
Progress: 2600/4000
Progress: 2700/4000
Progress: 2800/4000
Progress: 2900/4000
Progress: 3000/4000
Progress: 3100/4000
Progress: 3200/4000
Progress: 3300/4000
Progress: 3400/4000
✅ Processing complete (NO ROW LOSS GUARANTEED)
Input rows: 4000
Accounted rows: 4000


## The outputed Data seperated to only keep the merged records the unmerged records go through a manual checking phase to identify the mismatches
But there was no any mismatches found since i executed if any one model's similarity score above the threshold, it will merge.

In [8]:
def count_unique_ids(id_string):
    if pd.isna (id_string):
        return 0
    ids = [x.strip() for x in str(id_string).split(",") if x.strip()]
    return len(set(ids))

merged_df ["unique_id_count"] = merged_df["E NO"]. apply (count_unique_ids)

# Keep only merged records (≥ 2 unique registered IDs)
merged_df = merged_df[merged_df["unique_id_count"] >= 2]

# Optional: drop helper column
merged_df = merged_df.drop(columns=["unique_id_count"])
merged_df. to_excel("employer_merged_only_results.xlsx", index=False)

## COMPUTED TO ONLY RECEIVE THE ID1, ID, NAME1 AND NAME2 TO CREATE THE DATA VALIDATION, FOR THE TRUE AND FALSE MERGE LABELLING PROCESS

In [15]:
import pandas as pd

# Read your CSV or Excel file
df = pd.read_excel("C:\\Users\\USER\\Desktop\\Data cleaning (NLP)\\Output\\FUZZY_ALO_MODELS\\employer_merged_only_results.xlsx", sheet_name='evaluation_true_label')  # adjust if Excel: pd.read_excel("file.xlsx")

# Function to split IDs and names
def split_ids_names(row):
    ids = row[0].split(",")[:2]   # first 2 IDs
    names = row[1].split(",")[:2] # first 2 names
    state = row[2]
    # Fill missing if less than 2
    while len(ids) < 2:
        ids.append('')
    while len(names) < 2:
        names.append('')
    return pd.Series([ids[0], ids[1], names[0], names[1], state])

# Apply function
new_df = df.apply(split_ids_names, axis=1)
new_df.columns = ["ID1", "ID2", "NAME1", "NAME2", "STATE"]

# Save to Excel
new_df.to_excel("merged_split2.xlsx", index=False)


C:\Users\USER\AppData\Local\Temp\ipykernel_30892\4120659691.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ids = row[0].split(",")[:2]   # first 2 IDs
C:\Users\USER\AppData\Local\Temp\ipykernel_30892\4120659691.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  names = row[1].split(",")[:2] # first 2 names
C:\Users\USER\AppData\Local\Temp\ipykernel_30892\4120659691.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  state = r

In [22]:
# Read your CSV or Excel file
df = pd.read_excel("C:\\Users\\USER\\Desktop\\Data cleaning (NLP)\\Output\\FUZZY_ALO_MODELS\\FINAL_EVALUATION_DATA.xlsx")  # adjust if Excel: pd.read_excel("file.xlsx")


# -----------------------------
# Helper function: Normalize name
# -----------------------------
def normalize_name(name):
    """Normalize name for comparison"""
    if pd.isna(name):
        return ""
    return str(name).lower().strip()

# -----------------------------
# 1. Clean unwanted company suffixes
# -----------------------------
def clean_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower().strip()

    # Remove common business suffixes
    patterns = [
        r"\bsdn\.?\s*bhd\.?\b",
        r"\bsendiran\s*berhad\b",
        r"\bs/b\b",
        r"\bsdn\b",
        r"\bbhd\b",
    ]

    for p in patterns:
        name = re.sub(p, "", name)

    # Remove extra spaces
    name = re.sub(r"\s+", " ", name).strip()
    
    # REMOVE punctuation: . / ( ) , -
    name = re.sub(r"[./():,\-&@]", " ", name)

    return name

def clean_employer_id(eid):
    if pd.isna(eid):
        return ""

    eid = str(eid).lower().strip()

    # 🔹 Remove punctuation from ID
    eid = re.sub(r"[.,()/\-]", "", eid)

    # Remove extra spaces
    eid = re.sub(r"\s+", "", eid)

    return eid

df["NAME1"] = df["NAME1"].apply(clean_name)
df["NAME2"] = df["NAME2"].apply(clean_name)

df["ID1"] = df["ID1"].apply(clean_employer_id)
df["ID2"] = df["ID2"].apply(clean_employer_id)

# Save to a new Excel file
output_file = "FINAL_EVALUATION_DATA_CLEANED.xlsx"

df.to_excel(output_file, index=False)

print(f"File saved successfully as: {output_file}")


File saved successfully as: FINAL_EVALUATION_DATA_CLEANED.xlsx


## DATA VALIDATION WHICH CREATED TO THE OUTPUTED RECORD AND BASED ON THE ORIGINAL DATASET OF EWS (LABELLLED TRUE FOR THE REAL MERGEABLE DUPLICATES)

### EXECUTED TO DROP ALL THE DUPLICATE ROWS 
This data used for evaluation by computing back the scores and Move for Similarity models performance to choose the best moel.

In [26]:
df = pd.read_excel("C:\\Users\\USER\\Desktop\\Data cleaning (NLP)\\Output\\FUZZY_ALO_MODELS\\FINAL_EVALUATION_DATA.xlsx")  # adjust if Excel: pd.read_excel("file.xlsx")


# Remove rows where ID1 == ID2
df_filtered = df[df["ID1"] != df["ID2"]].copy()
print("Original rows:", len(df))
print("Rows after removing duplicate IDs:", len(df_filtered))
print("Rows removed:", len(df) - len(df_filtered))
df_filtered.to_excel(
    "FINALL_EVALUATION_DATA_CLEANED_NO_SELF_MATCH.xlsx",
    index=False
)


Original rows: 456
Rows after removing duplicate IDs: 346
Rows removed: 110
